# 📓 Notebook 1 — Formalizando nuestro Modelo Mental

**Curso:** Satélites y Sims: ¡Juega con el futuro de tu ciudad!
**Sección 2 — Modelación y Gamificación** · Día 2 (tarde)

### Objetivo de hoy
Construir un modelo mental del sistema ecológico-urbano identificando variables y relaciones, y **formalizarlo**: definir variables, sus rangos y relaciones matemáticas simples.

Esta libreta es el puente entre el **diagrama causal** que dibujaron en el pizarrón/Miro con su equipo y el **código en Python** que escribirán mañana. No necesitan saber programar todavía: hoy solo *organizamos ideas* en una tabla, casi como llenar una receta.

> 💡 Tip: tengan a la mano una foto de su diagrama causal (pizarrón o Miro) mientras trabajan aquí.


## 0. Conectemos con lo que ya saben

Durante la Sección 1 pasaron dos días analizando imágenes satelitales reales de su ciudad: calcularon NDVI, compararon fechas y detectaron cambios en vegetación, construcción y cuerpos de agua.

**✏️ Antes de seguir, respondan como equipo (editen esta celda o escriban abajo en una nueva):**

1. De los cambios que detectaron en sus imágenes satelitales, ¿cuáles creen que están **conectados entre sí**? (por ejemplo: ¿la pérdida de vegetación que vieron podría estar relacionada con el aumento de construcción?)
2. ¿Qué tan bien creen que un conjunto de flechas y variables puede representar algo tan complejo como una ciudad? ¿Qué se les ocurre que se podría perder al simplificarlo así?

*Guarden sus respuestas — al final de este notebook regresaremos a esta misma pregunta.*


## 1. Repaso: ¿qué es un diagrama causal?

Un **diagrama causal** conecta variables de un sistema con flechas que indican cómo una afecta a otra:

- Flecha **`+`** (misma dirección): si la variable de origen sube, la de destino también sube (o si baja, también baja).
- Flecha **`-`** (dirección opuesta): si la variable de origen sube, la de destino baja (y viceversa).

Ejemplo:

```
Construcción ──(+)──> Población
Construcción ──(-)──> Vegetación
Vegetación ──(-)──> Contaminación
Población ──(+)──> Contaminación
```

Un **ciclo de retroalimentación** ocurre cuando, siguiendo las flechas, regresamos a la variable de inicio. Puede ser:

- **Ciclo de refuerzo**: amplifica el cambio (ej. más población → más construcción → más población).
- **Ciclo de balance**: contrarresta el cambio (ej. más contaminación → menos calidad de vida → menos población → menos contaminación).

**Actividad en equipo:** identifiquen al menos un ciclo de refuerzo y uno de balance en su diagrama antes de continuar.

**🤔 Antes de continuar — predicción:** miren su diagrama de pizarrón/Miro y, sin usar todavía la herramienta automática de la Sección 4, cuenten a mano cuántos ciclos creen que tiene. Anótenlo: `Predicción: ____ ciclos`. Lo compararemos más adelante con lo que detecte el código.


In [ ]:
# Ejecuta esta celda para preparar las herramientas que usaremos hoy.
# No necesitas entender todo el código todavía — solo corre la celda (Shift + Enter).

import json
from dataclasses import dataclass, field
from typing import List

print("Listo. Herramientas cargadas para formalizar el modelo. 🚀")


## 2. Lista de variables de nuestro sistema

Junto con tu equipo, llenen la lista de variables identificadas en la lluvia de ideas (vegetación, construcción, contaminación, agua, población, etc.).

Para cada variable definan:

| Campo | Qué significa | Ejemplo |
|---|---|---|
| `nombre` | Nombre corto de la variable | `"vegetacion"` |
| `descripcion` | Qué representa | `"Cobertura vegetal de la zona (%)"` |
| `unidad` | Cómo se mide | `"% de cobertura"` |
| `rango` | Valor mínimo y máximo posible | `(0, 100)` |
| `valor_inicial` | Valor con el que arranca la simulación | `60` |

Edita la lista `variables` de la celda de abajo — **agrega, quita o cambia** las variables según su diagrama causal. El ejemplo trae 5 variables típicas del sistema ecológico-urbano; bórralas si su equipo eligió otras.

**🤔 Antes de programar — predicción:** sin ver todavía la lista de ejemplo de la celda de abajo, escriban en un papel las 3 variables que su equipo considera **más importantes** para explicar los cambios en su zona. ¿Por qué esas 3 y no otras?


In [ ]:
# ✏️ EDITA esta lista con las variables de TU equipo.
# Cada variable es un diccionario con: nombre, descripcion, unidad, rango (min, max) y valor_inicial.

variables = [
    {
        "nombre": "vegetacion",
        "descripcion": "Cobertura vegetal de la zona",
        "unidad": "% de cobertura",
        "rango": (0, 100),
        "valor_inicial": 60,
    },
    {
        "nombre": "construccion",
        "descripcion": "Área construida / urbanizada",
        "unidad": "% de la zona",
        "rango": (0, 100),
        "valor_inicial": 30,
    },
    {
        "nombre": "poblacion",
        "descripcion": "Número de habitantes (en miles)",
        "unidad": "miles de habitantes",
        "rango": (0, 500),
        "valor_inicial": 100,
    },
    {
        "nombre": "contaminacion",
        "descripcion": "Nivel de contaminación (aire/agua)",
        "unidad": "índice 0-100",
        "rango": (0, 100),
        "valor_inicial": 20,
    },
    {
        "nombre": "agua",
        "descripcion": "Disponibilidad / calidad de cuerpos de agua",
        "unidad": "índice 0-100",
        "rango": (0, 100),
        "valor_inicial": 70,
    },
]

for v in variables:
    print(f"- {v['nombre']}: {v['descripcion']} (rango {v['rango']}, inicial={v['valor_inicial']})")


### 🔍 Observen y reflexionen

Comparen la lista final de `variables` (la que acaban de editar) con las 3 variables que predijeron a mano.

**✏️ Escriban aquí sus respuestas como equipo** *(reemplacen el texto en cursiva)*:

- ¿Sus 3 variables predichas están en la lista final? *(reemplazar)*
- ¿Agregaron alguna variable que no habían pensado antes? ¿Cuál y por qué terminó siendo importante? *(reemplazar)*
- ¿Alguna variable de su predicción terminó fuera de la lista final? ¿Por qué decidieron quitarla? *(reemplazar)*


## 3. Relaciones entre variables

Ahora traduzcamos las **flechas** de su diagrama causal a una tabla de relaciones. Cada relación dice:

- `origen`: variable que causa el cambio
- `destino`: variable que recibe el efecto
- `signo`: `+1` si van en la misma dirección, `-1` si van en dirección opuesta
- `fuerza`: qué tan fuerte es el efecto, de `0.0` (nulo) a `1.0` (muy fuerte) — esto es una estimación del equipo, ¡no hay una respuesta única!
- `explicacion`: una frase que justifique la relación (útil para su presentación final)

Esto es exactamente lo que anotaron como flechas en el pizarrón/Miro — solo lo estamos poniendo en una tabla ordenada.


In [ ]:
# ✏️ EDITA esta lista con las relaciones (flechas) de TU diagrama causal.

relaciones = [
    {"origen": "construccion", "destino": "vegetacion", "signo": -1, "fuerza": 0.4,
     "explicacion": "Más construcción reduce el área con vegetación."},
    {"origen": "construccion", "destino": "poblacion", "signo": +1, "fuerza": 0.3,
     "explicacion": "Más vivienda construida permite que llegue más población."},
    {"origen": "poblacion", "destino": "contaminacion", "signo": +1, "fuerza": 0.3,
     "explicacion": "Más habitantes generan más residuos y emisiones."},
    {"origen": "vegetacion", "destino": "contaminacion", "signo": -1, "fuerza": 0.35,
     "explicacion": "La vegetación absorbe contaminantes y regula el aire."},
    {"origen": "contaminacion", "destino": "agua", "signo": -1, "fuerza": 0.3,
     "explicacion": "La contaminación reduce la calidad del agua disponible."},
    {"origen": "agua", "destino": "vegetacion", "signo": +1, "fuerza": 0.25,
     "explicacion": "Más agua disponible favorece el crecimiento de vegetación."},
]

nombres_validos = {v["nombre"] for v in variables}
for r in relaciones:
    assert r["origen"] in nombres_validos, f"'{r['origen']}' no está en tu lista de variables"
    assert r["destino"] in nombres_validos, f"'{r['destino']}' no está en tu lista de variables"
    assert r["signo"] in (1, -1), "signo debe ser +1 o -1"
    assert 0.0 <= r["fuerza"] <= 1.0, "fuerza debe estar entre 0.0 y 1.0"

print(f"{len(relaciones)} relaciones registradas y validadas ✅")


### 🔍 Observen y reflexionen

**✏️ Discutan en equipo y escriban su respuesta:**

- ¿Todos en el equipo hubieran puesto el mismo `signo` (+1 o -1) en cada relación? ¿En cuál relación hubo más desacuerdo y cómo lo resolvieron? *(reemplazar)*
- La `fuerza` de cada relación es una estimación subjetiva de ustedes, no un dato medido. ¿Qué evidencia de las imágenes satelitales de la Sección 1 podría ayudarles a justificar mejor alguna de sus fuerzas? *(reemplazar)*


## 4. Detectar ciclos de retroalimentación (automático)

Corre la celda de abajo — recorre tus relaciones y encuentra ciclos cortos automáticamente, calculando si son de **refuerzo** (signo neto positivo) o de **balance** (signo neto negativo). Esto les ayuda a verificar que el ciclo que identificaron a mano en el pizarrón también aparece aquí.


In [ ]:
def encontrar_ciclos(relaciones, max_len=4):
    grafo = {}
    for r in relaciones:
        grafo.setdefault(r["origen"], []).append(r)

    ciclos = []

    def dfs(inicio, actual, signo_acumulado, camino, visitados_en_camino):
        if len(camino) > max_len:
            return
        for r in grafo.get(actual, []):
            nuevo_signo = signo_acumulado * r["signo"]
            if r["destino"] == inicio and len(camino) >= 1:
                ciclos.append((camino + [r], nuevo_signo))
            elif r["destino"] not in visitados_en_camino:
                dfs(inicio, r["destino"], nuevo_signo, camino + [r], visitados_en_camino | {r["destino"]})

    for var in {r["origen"] for r in relaciones}:
        dfs(var, var, 1, [], {var})

    # quitar duplicados (mismo ciclo recorrido desde distinto punto de partida)
    vistos = set()
    unicos = []
    for camino, signo in ciclos:
        clave = tuple(sorted((r["origen"], r["destino"]) for r in camino))
        if clave not in vistos:
            vistos.add(clave)
            unicos.append((camino, signo))
    return unicos

ciclos = encontrar_ciclos(relaciones)

if not ciclos:
    print("No se detectaron ciclos con las relaciones actuales.")
    print("Revisen si falta alguna flecha de su diagrama (a veces el ciclo se cierra con una relación que olvidaron capturar).")
else:
    for i, (camino, signo) in enumerate(ciclos, 1):
        tipo = "🔁 REFUERZO" if signo > 0 else "⚖️ BALANCE"
        ruta = " → ".join([camino[0]["origen"]] + [r["destino"] for r in camino])
        print(f"Ciclo {i} ({tipo}): {ruta}")


### 🔍 Observen y reflexionen

Comparen el número de ciclos que detectó el código con su predicción de la Sección 1 de este notebook.

**✏️ Escriban aquí su respuesta:**

- ¿Coincidió el número? Si no coincidió, ¿el código encontró ciclos que no habían visto a mano, o al revés? *(reemplazar)*
- Elijan uno de los ciclos detectados: en sus propias palabras, expliquen qué pasaría en la realidad si esa cadena de causas se repitiera varias veces seguidas. *(reemplazar)*
- ¿Un ciclo de refuerzo es necesariamente "malo" y uno de balance necesariamente "bueno"? Den un ejemplo de su propio modelo que muestre que no siempre es tan simple. *(reemplazar)*


## 5. Relaciones matemáticas simples

Por último, escribamos **una ecuación sencilla** por variable que describa cómo cambia en cada "paso" (turno/año) de la simulación, usando las relaciones de arriba. No tiene que ser perfecta — mañana la vamos a programar y ajustar.

Plantilla general:

```
nueva_variable = variable_actual + suma_de_efectos_de_sus_causas
```

Por ejemplo, para `contaminacion`:

```
nueva_contaminacion = contaminacion + (0.3 * poblacion_normalizada) - (0.35 * vegetacion_normalizada)
```

Completen la tabla de abajo con una fórmula en palabras (no hace falta código Python todavía) para cada variable.


In [ ]:
# ✏️ EDITA: escribe en palabras la fórmula de cambio para cada variable.
# Esto es tu "borrador matemático" — mañana lo convertimos a código Python real.

formulas_en_palabras = {
    "vegetacion": "vegetacion = vegetacion - 0.4*construccion_normalizada + 0.25*agua_normalizada",
    "construccion": "construccion = construccion + tasa_construccion_fija (decisión del jugador)",
    "poblacion": "poblacion = poblacion + 0.3*construccion_normalizada * tasa_crecimiento",
    "contaminacion": "contaminacion = contaminacion + 0.3*poblacion_normalizada - 0.35*vegetacion_normalizada",
    "agua": "agua = agua - 0.3*contaminacion_normalizada + 0.25*vegetacion_normalizada",
}

for var, formula in formulas_en_palabras.items():
    print(f"{var}:\n   {formula}\n")


## 6. Guardar el modelo formalizado

Esta celda guarda todo lo que definieron en un archivo `modelo_mental.json`. **Mañana en el Notebook 2 vamos a cargar este archivo** para construir la simulación en Python directamente a partir de su modelo — ¡no hay que volver a escribir las variables!


In [ ]:
modelo = {
    "equipo": "NOMBRE_DE_TU_EQUIPO",   # ✏️ cambia esto por el nombre de tu equipo
    "zona_de_estudio": "NOMBRE_DE_TU_ZONA",  # ✏️ ej. 'Centro de Mérida'
    "variables": variables,
    "relaciones": relaciones,
    "formulas_en_palabras": formulas_en_palabras,
}

with open("modelo_mental.json", "w", encoding="utf-8") as f:
    json.dump(modelo, f, ensure_ascii=False, indent=2)

print("✅ Modelo guardado en 'modelo_mental.json'")
print(f"   Equipo: {modelo['equipo']}")
print(f"   Variables: {len(variables)} | Relaciones: {len(relaciones)} | Ciclos detectados: {len(ciclos)}")


## 🧭 Bitácora de aprendizaje

Cierren la sesión escribiendo, como equipo, sus propias respuestas — no hay una única respuesta correcta, lo importante es que reflejen lo que de verdad pensaron y discutieron hoy.

**✏️ Completen cada punto:**

1. **Regresen a la pregunta de la Sección 0** (¿qué se pierde al simplificar una ciudad en variables y flechas?). Ahora que formalizaron su propio modelo, ¿cambiaría su respuesta? ¿Por qué? *(reemplazar)*
2. En sus propias palabras — sin copiar la definición de la Sección 1 — ¿qué es un modelo mental y para qué sirve formalizarlo antes de programar? *(reemplazar)*
3. ¿Qué variable de su modelo tiene más flechas entrando? ¿Por qué creen que es tan influyente en su sistema? *(reemplazar)*
4. Si tuvieran que simplificar su modelo a solo 3 variables, ¿cuáles elegirían y qué perderían al hacerlo? *(reemplazar)*
5. ¿Qué escenario de política (visto en la presentación de hoy) cambiaría más drásticamente su modelo? ¿Qué relación específica se vería más afectada? *(reemplazar)*
6. ¿Qué parte de su modelo les genera más dudas o menos confianza? Es la pregunta que le van a hacer a su mentor o instructor. *(reemplazar)*

## ✅ Producto esperado de esta sesión

- [ ] Diagrama causal en pizarrón/Miro con variables y relaciones (foto guardada por el equipo)
- [ ] Este notebook completado: lista de `variables`, `relaciones` y `formulas_en_palabras`
- [ ] Al menos un ciclo de refuerzo y uno de balance identificados
- [ ] Archivo `modelo_mental.json` generado y guardado (lo usarán mañana)
- [ ] Bitácora de aprendizaje completa con las 6 reflexiones de esta sección

**Mañana:** Notebook 2 — traduciremos este modelo a una simulación numérica funcional en Python. 🐍
